# PyTorch for Transformers and LLMs

**PyTorch Interview Preparation · Notebook 10 · Estimated study time: 3–4 hours**

This notebook is about the PyTorch syntax, tensor shapes, and implementation patterns behind Transformers and decoder-style language models. It deliberately uses tiny synthetic tensors, runs on CPU, needs no internet, and uses only PyTorch.

The learning loop is: **see a shape → predict the next shape → write the operation → run it → inspect it → explain it aloud**.

Difficulty labels: 🟢 Easy · 🟡 Medium · 🔴 Hard / optional


## 1. Setup


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
print("PyTorch:", torch.__version__)
print("Device: CPU")


## 2. Transformer Tensor Shapes

### 🔥 Interview Essential — memorize the batch-first convention

| Symbol | Meaning |
|---|---|
| `B` | batch size |
| `T` | sequence length |
| `D` | hidden / embedding dimension |
| `V` | vocabulary size |
| `H` | number of attention heads |
| `Dh` | head dimension, `D / H` |

The main hidden-state shape is **`(batch, sequence, hidden_size)`**, abbreviated `(B, T, D)`. Thus `(32, 128, 768)` means 32 sequences, 128 tokens per sequence, and 768 features per token.

```text
batch
  ↓
[
  sequence 1: token token token ...
  sequence 2: token token token ...
  ...
]

each token → vector of hidden_size values
```

The batch and sequence axes organize data. The final axis stores each token's features.


In [ ]:
B, T, D = 2, 5, 8
x = torch.randn(B, T, D)
assert x.shape == (2, 5, 8)
print("hidden states:", x.shape)


### Shape flashcards — 🟢 Easy

1. `x.shape = (16, 128, 512)`: identify B, T, D.

<details><summary>Answer</summary>B=16, T=128, D=512</details>
2. `x.shape = (4, 20, 64)`: how many token vectors?

<details><summary>Answer</summary>4×20=80</details>
3. Tokens have shape `(32, 100)`. Identify B and T.

<details><summary>Answer</summary>B=32, T=100</details>
4. Logits have shape `(8, 50, 30000)`. Identify V.

<details><summary>Answer</summary>V=30000</details>
5. Scores have shape `(2, 8, 64, 64)`. Identify H and T.

<details><summary>Answer</summary>H=8, T=64</details>
6. A split-head tensor is `(3, 12, 128, 64)`. What is D?

<details><summary>Answer</summary>12×64=768</details>
7. `B=6,T=10,D=96`: write the hidden-state shape.

<details><summary>Answer</summary>(6,10,96)</details>
8. Embeddings are `(5, 7, 32)`. How many features per token?

<details><summary>Answer</summary>32</details>
9. Targets are `(8, 127)`. How many token labels?

<details><summary>Answer</summary>8×127=1016</details>
10. Per-head values are `(4, 4, 20, 16)`. What is combined shape?

<details><summary>Answer</summary>(4,20,64)</details>


## 3. Token IDs

Text is converted to integer token IDs before entering a Transformer. These numbers are indices into a vocabulary; **they are not semantic vectors yet**. Input IDs normally use `torch.long`, which `nn.Embedding` expects.

Vocabulary size is the number of possible IDs. If `vocab_size = 50_000`, valid IDs are normally `0` through `49_999`. Tokenizer algorithms are outside this notebook's scope.


In [ ]:
tokens = torch.tensor([
    [12, 45, 983, 5],
    [92, 17, 5, 0],
])
print(tokens)
print("shape:", tokens.shape, "dtype:", tokens.dtype)
assert tokens.shape == (2, 4) and tokens.dtype == torch.long


## 4. Token Embeddings

### 🔥 Interview Essential

`nn.Embedding(V, D)` is a learned lookup table with weight shape `(V, D)`. It maps IDs `(B, T)` to vectors `(B, T, D)`.

```text
token IDs (B,T) → Embedding → token vectors (B,T,D)
```


In [ ]:
vocab_size, D = 50_000, 768
embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=D)
small_tokens = torch.randint(0, vocab_size, (2, 6))
x_emb = embedding(small_tokens)
print("IDs:", small_tokens.shape, "embeddings:", x_emb.shape)
print("weight:", embedding.weight.shape)
assert x_emb.shape == (2, 6, 768)


### Embedding parameter-count practice — 🟢 Easy

An embedding has `V × D` parameters and no separate bias.

1. `Embedding(50_000, 768)` → **38,400,000**
2. `Embedding(10_000, 128)` → **1,280,000**
3. `Embedding(128, 64)` → **8,192**
4. If the weight is `(30_000, 512)`, what are `V` and `D`? → **30,000 and 512**
5. Does batch size alter the embedding parameter count? → **No.**


In [ ]:
assert embedding.weight.numel() == vocab_size * D
print(f"embedding parameters: {embedding.weight.numel():,}")


## 5. Positional Information

Self-attention alone does not inherently know token order, so inputs need positional information. Two broad approaches are learned positional embeddings and fixed positional encodings. We use learned positions here; RoPE is intentionally deferred.


In [ ]:
max_seq_len, D = 512, 128
position_embedding = nn.Embedding(max_seq_len, D)
positions = torch.arange(10)
p = position_embedding(positions)
assert p.shape == (10, 128)
print(positions, p.shape)


Add token and position embeddings. Broadcasting expands `(T,D)` across the batch:

```text
x: (B,T,D)
p:   (T,D)
--------------
     (B,T,D)
```


In [ ]:
B, T, D = 4, 10, 128
ids = torch.randint(0, 10_000, (B, T))
token_embedding = nn.Embedding(10_000, D)
position_embedding = nn.Embedding(512, D)
x = token_embedding(ids)
p = position_embedding(torch.arange(T))
x = x + p
assert x.shape == (4, 10, 128)
print(x.shape)


### Exercise — 🟢 Easy

Create learned token and position embeddings for `batch=8`, `sequence=20`, `hidden=64`. Validate `(8,20,64)`.


In [ ]:
# One solution
B_ex, T_ex, D_ex = 8, 20, 64
ids_ex = torch.randint(0, 1_000, (B_ex, T_ex))
tok_ex = nn.Embedding(1_000, D_ex)
pos_ex = nn.Embedding(64, D_ex)
x_ex = tok_ex(ids_ex) + pos_ex(torch.arange(T_ex))
assert x_ex.shape == (8, 20, 64)


## 6. Query, Key, and Value

### 🔥 Interview Essential

Each token representation is linearly projected into query `Q`, key `K`, and value `V` representations. A useful—but not literal—intuition is:

- Query: what information might this position look for?
- Key: what information might this position make available for matching?
- Value: what information is mixed into the output?

For self-attention, all three derive from the same input. Projection preserves `(B,T,D)` when each layer is `Linear(D,D)`.


In [ ]:
B, T, D = 2, 4, 8
x = torch.randn(B, T, D)
q_proj, k_proj, v_proj = nn.Linear(D, D), nn.Linear(D, D), nn.Linear(D, D)
Q, K, V = q_proj(x), k_proj(x), v_proj(x)
assert Q.shape == K.shape == V.shape == (B, T, D)
print("Q/K/V:", Q.shape, K.shape, V.shape)


## 7. Scaled Dot-Product Attention

### 🔥 Interview Essential · 🔥 Very Common Interview Question

First learn attention without heads:

```python
scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
weights = torch.softmax(scores, dim=-1)
output = weights @ V
```

`K.transpose(-2,-1)` changes `(B,T,D)` to `(B,D,T)`, enabling `(T,D) @ (D,T) → (T,T)`. Every query position gets a score against every key position.

The scale factor is `sqrt(d_k)`. Dot-product magnitudes tend to grow with key/query dimension; scaling keeps scores more controlled before softmax.

Softmax uses `dim=-1`: for each query, normalize across key positions. The weights on that axis should sum to one.


In [ ]:
raw_scores = Q @ K.transpose(-2, -1)
scores = raw_scores / math.sqrt(D)
weights = torch.softmax(scores, dim=-1)
attention_output = weights @ V
print("scores:", scores.shape, "weights:", weights.shape, "output:", attention_output.shape)
print("row sums:\n", weights.sum(dim=-1))
assert scores.shape == (B, T, T)
assert attention_output.shape == (B, T, D)
assert torch.allclose(weights.sum(dim=-1), torch.ones(B, T), atol=1e-6)


### Implement `scaled_dot_product_attention()` — 🟡 Medium

The mask convention in this helper is boolean `True = blocked`. The last query/key axes may differ, so the returned score/weight shape is `(..., T_query, T_key)`.


In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    if Q.size(-1) != K.size(-1):
        raise ValueError("Q and K must have the same key dimension")
    if K.size(-2) != V.size(-2):
        raise ValueError("K and V must have the same context length")
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(Q.size(-1))
    if mask is not None:
        if mask.dtype != torch.bool:
            raise TypeError("this helper expects a boolean mask")
        scores = scores.masked_fill(mask, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    output = weights @ V
    assert output.shape[:-1] == Q.shape[:-1]
    assert output.size(-1) == V.size(-1)
    return output, weights


function_output, function_weights = scaled_dot_product_attention(Q, K, V)
assert function_output.shape == (B, T, D)
assert function_weights.shape == (B, T, T)


```text
Q × Kᵀ → scores → scale → softmax → attention weights → × V → output

        keys
       T tokens
     ┌─────────┐
Q T  │  T × T  │
     └─────────┘
```


### Attention score-shape drills — 🟡 Medium

1. Q `(2,4,8)`, K `(2,4,8)` → <details><summary>Answer</summary>scores `(2,4,4)`</details>
2. Q `(16,128,64)`, K `(16,128,64)` → <details><summary>Answer</summary>`(16,128,128)`</details>
3. Q `(1,10,32)`, K `(1,20,32)` → <details><summary>Answer</summary>`(1,10,20)`</details>
4. Q `(8,5,16)`, K `(8,7,16)` → <details><summary>Answer</summary>`(8,5,7)`</details>
5. Q `(3,2,4)`, K `(3,9,4)` → <details><summary>Answer</summary>`(3,2,9)`</details>
6. weights `(4,6,6)`, V `(4,6,12)` → <details><summary>Answer</summary>output `(4,6,12)`</details>
7. weights `(2,3,7)`, V `(2,7,5)` → <details><summary>Answer</summary>output `(2,3,5)`</details>
8. K `(5,11,24)` after transpose → <details><summary>Answer</summary>`(5,24,11)`</details>
9. scores `(7,13,13)`, softmax `dim=-1` → <details><summary>Answer</summary>unchanged `(7,13,13)`</details>
10. Q length 12, context length 30 → <details><summary>Answer</summary>score matrix last axes `(12,30)`</details>


### Write from memory

Turn `K` from `(B,T,D)` into `(B,D,T)`, then compute scaled scores. Say aloud why the transpose is needed.


In [ ]:
K_t = K.transpose(-2, -1)
scores_memory = (Q @ K_t) / math.sqrt(Q.size(-1))
assert scores_memory.shape == (B, T, T)


## 8. Attention Score Shapes

The score matrix contains pairwise query–key compatibility scores. Its final two axes are always `query_length × key_length`; they happen to both be `T` in ordinary self-attention.


## 9. Softmax in Attention

`softmax(scores, dim=-1)` makes each query row a distribution over keys. Using a batch, head, or query axis answers a different question and is usually a bug.


## 10. Causal Masks

### 🔥 Interview Essential for LLMs

At autoregressive position `t`, a decoder must not see future tokens:

```text
✓ × × ×
✓ ✓ × ×
✓ ✓ ✓ ×
✓ ✓ ✓ ✓
```

Here `True` means “block.” `torch.triu(..., diagonal=1)` selects positions strictly above the diagonal. Filling those scores with `-inf` makes their softmax probabilities zero.


In [ ]:
T = 4
causal_mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
print(causal_mask)
masked_scores = scores.masked_fill(causal_mask, float("-inf"))
causal_weights = torch.softmax(masked_scores, dim=-1)
print(causal_weights[0])
assert causal_mask.shape == (4, 4)
assert torch.all(causal_weights.masked_select(causal_mask) == 0)


### Causal-mask exercise — 🟡 Medium

Build a `(6,6)` mask. Verify that the upper triangle above the diagonal is masked and the diagonal is available.


In [ ]:
mask6 = torch.triu(torch.ones(6, 6, dtype=torch.bool), diagonal=1)
assert mask6.shape == (6, 6)
assert not mask6.diagonal().any()
assert mask6.sum().item() == 6 * 5 // 2


## 11. Padding Masks

Padding masks prevent attention from using padded key positions. They have shape `(B,T)`; causal masks usually have shape `(T,T)`.

| Mask | Purpose |
|---|---|
| **causal mask** | hide future positions |
| **padding mask** | hide padding positions in each batch item |

### 🔥 Interview Essential — these solve different problems and may be used together.


In [ ]:
tokens_with_pad = torch.tensor([[5, 7, 9, 0, 0], [8, 3, 4, 2, 0]])
padding_mask = tokens_with_pad == 0
print(padding_mask)
assert padding_mask.shape == (2, 5)


## 12. Self-Attention

### 🔥 Interview Essential

Self-attention means query, key, and value are derived from the same sequence: `attention(x, x, x)`. Encoder blocks and decoder blocks both use it, although decoder self-attention is normally causal.


## 13. Cross-Attention

In cross-attention, queries come from one sequence while keys and values come from another:

```text
decoder hidden states → Q:   (B,T_query,D)
encoder hidden states → K/V: (B,T_context,D)
scores:                      (B,T_query,T_context)
```

Shape checks:

1. `Q (4,10,64)`, `K/V (4,20,64)` → scores **`(4,10,20)`**.
2. Output after weights × V → **`(4,10,64)`**.
3. Which length determines output positions? → **query length**.
4. Is `attention(decoder, encoder, encoder)` self- or cross-attention? → **cross-attention**.


## 14. Multi-Head Attention

### 🔥 Interview Essential

Multi-head attention divides `D` into `H` parallel heads. Each head gets `Dh = D/H` features and its own attention matrix. The critical rule is:

```text
D % H == 0
```

Examples: `768/12=64`, `512/8=64`; `1000/12` does not divide evenly.


## 15. Splitting Into Heads

### 🔥 Interview Essential Shape

```text
Before:          (B,T,D), where D=H×Dh
After view:      (B,T,H,Dh)
After transpose: (B,H,T,Dh)
```

Placing heads before sequence makes batched attention natural. Prefer `reshape` when contiguity is uncertain; `view` requires compatible memory layout.


In [ ]:
B, T, D, H = 2, 5, 32, 4
Dh = D // H
q_full = torch.randn(B, T, D)
q_heads = q_full.view(B, T, H, Dh).transpose(1, 2)
assert q_heads.shape == (B, H, T, Dh)
print(q_heads.shape)


### Split-head reshape drills — 🟡 Medium

1. `(B,T,D)=(2,5,32)`, `H=4` → `Dh=8`, result `(2,4,5,8)`.
2. `(B,T,D)=(8,128,512)`, `H=8` → `Dh=64`, result `(8,8,128,64)`.
3. `(B,T,D)=(1,10,768)`, `H=12` → `Dh=64`, result `(1,12,10,64)`.
4. `(B,T,D)=(4,20,64)`, `H=4` → `Dh=16`, result `(4,4,20,16)`.
5. `(B,T,D)=(16,50,256)`, `H=16` → `Dh=16`, result `(16,16,50,16)`.
6. `(B,T,D)=(3,7,96)`, `H=6` → `Dh=16`, result `(3,6,7,16)`.
7. `(B,T,D)=(2,100,128)`, `H=8` → `Dh=16`, result `(2,8,100,16)`.
8. `(B,T,D)=(5,9,120)`, `H=10` → `Dh=12`, result `(5,10,9,12)`.
9. `(B,T,D)=(6,12,48)`, `H=3` → `Dh=16`, result `(6,3,12,16)`.
10. `(B,T,D)=(1,32,1024)`, `H=16` → `Dh=64`, result `(1,16,32,64)`.


## 16. Combining Heads

After per-head attention:

```text
(B,H,T,Dh) → transpose → (B,T,H,Dh) → reshape → (B,T,D)
```

An output projection `Linear(D,D)` then mixes information across heads.


In [ ]:
combined = q_heads.transpose(1, 2).reshape(B, T, D)
assert combined.shape == (B, T, D)


## 17. Manual Multi-Head Attention — 🔴 Hard / Optional


In [ ]:
class SimpleMultiHeadAttention(nn.Module):
    def __init__(self, hidden_size, num_heads, dropout=0.0):
        super().__init__()
        if hidden_size % num_heads != 0:
            raise ValueError("hidden_size must be divisible by num_heads")
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.q_proj = nn.Linear(hidden_size, hidden_size)
        self.k_proj = nn.Linear(hidden_size, hidden_size)
        self.v_proj = nn.Linear(hidden_size, hidden_size)
        self.out_proj = nn.Linear(hidden_size, hidden_size)
        self.dropout = nn.Dropout(dropout)

    def _split(self, x):
        B, T, _ = x.shape
        return x.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, query, key=None, value=None, causal=False, padding_mask=None):
        key = query if key is None else key
        value = key if value is None else value
        Q, K, V = self._split(self.q_proj(query)), self._split(self.k_proj(key)), self._split(self.v_proj(value))
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if causal:
            Tq, Tk = scores.shape[-2:]
            if Tq != Tk:
                raise ValueError("This simple causal path expects equal query/key lengths")
            mask = torch.triu(torch.ones(Tq, Tk, device=scores.device, dtype=torch.bool), diagonal=1)
            scores = scores.masked_fill(mask, float("-inf"))
        if padding_mask is not None:
            scores = scores.masked_fill(padding_mask[:, None, None, :], float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        heads = weights @ V
        B, _, Tq, _ = heads.shape
        merged = heads.transpose(1, 2).reshape(B, Tq, self.hidden_size)
        return self.out_proj(merged), weights


manual_mha = SimpleMultiHeadAttention(32, 4)
manual_x = torch.randn(2, 6, 32)
manual_out, manual_weights = manual_mha(manual_x, causal=True)
assert manual_out.shape == (2, 6, 32)
assert manual_weights.shape == (2, 4, 6, 6)
print(manual_out.shape, manual_weights.shape)


## 18. `nn.MultiheadAttention`

### 🔥 Important PyTorch Syntax

With `batch_first=True`, inputs use `(B,T,D)`. The call signature is `attention(query,key,value)`: `attention(x,x,x)` is self-attention; `attention(decoder_states,encoder_states,encoder_states)` is cross-attention.

By default, returned attention weights are averaged across heads and have `(B,Tq,Tk)`. Set `average_attn_weights=False` for `(B,H,Tq,Tk)`.


In [ ]:
attention = nn.MultiheadAttention(embed_dim=128, num_heads=8, batch_first=True)
mha_x = torch.randn(4, 10, 128)
mha_output, avg_weights = attention(mha_x, mha_x, mha_x)
_, per_head_weights = attention(mha_x, mha_x, mha_x, average_attn_weights=False)
assert mha_output.shape == (4, 10, 128)
assert avg_weights.shape == (4, 10, 10)
assert per_head_weights.shape == (4, 8, 10, 10)
print(mha_output.shape, avg_weights.shape, per_head_weights.shape)


`attn_mask` expresses structural constraints such as causality, usually over query/key positions. `key_padding_mask` expresses padded key positions per batch item. Boolean `True` entries are blocked in these examples. Keep mask dtypes consistent when both are supplied.


In [ ]:
T = mha_x.size(1)
mha_causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
mha_padding_mask = torch.zeros(4, T, dtype=torch.bool)
mha_padding_mask[0, -2:] = True
masked_out, _ = attention(mha_x, mha_x, mha_x, attn_mask=mha_causal_mask, key_padding_mask=mha_padding_mask)
assert masked_out.shape == mha_x.shape


## 19. Residual Connections

### 🔥 Interview Essential

Residual connections add a sublayer output to its input: `x = x + attention_output`.

```text
x ─────────────────┐
│                  │
↓                  │
Attention          │
│                  │
└────── + ◄────────┘
        ↓
      output
```

They preserve information, improve gradient flow, and help deep networks train.


## 20. LayerNorm

### 🔥 Interview Essential

`nn.LayerNorm(D)` normalizes across the final hidden/features dimension for each token. Shape remains `(B,T,D)`.

- **Pre-norm:** `x = x + sublayer(norm(x))`
- **Post-norm:** `x = norm(x + sublayer(x))`

Modern implementations often use a pre-norm variant, but architectures differ. Unlike BatchNorm, LayerNorm does not rely on statistics aggregated over the batch.


In [ ]:
norm = nn.LayerNorm(128)
normed = norm(mha_x)
assert normed.shape == mha_x.shape


## 21. Feed-Forward Networks

### 🔥 Interview Essential

A common Transformer FFN is `D → 4D → D`. `nn.Linear` operates on the final axis, so it applies independently to every token without flattening batch and sequence.


In [ ]:
D = 128
ffn = nn.Sequential(nn.Linear(D, 4 * D), nn.GELU(), nn.Linear(4 * D, D))
ffn_out = ffn(mha_x)
assert ffn_out.shape == (4, 10, 128)


### FFN shape drills — 🟡 Medium

1. `(32,128,768)` through `Linear(768,3072)` → `(32,128,3072)`.
2. Then `Linear(3072,768)` → `(32,128,768)`.
3. `(4,20,64)` through `Linear(64,256)` → `(4,20,256)`.
4. Does sequence length change? → No.
5. Must `(B,T)` be flattened first? → No; doing so is unnecessary.


## 22. Building a Transformer Block


In [ ]:
class TransformerBlock(nn.Module):
    """A compact pre-norm, batch-first Transformer block."""
    def __init__(self, hidden_size, num_heads, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size)
        self.attn = nn.MultiheadAttention(hidden_size, num_heads, dropout=dropout, batch_first=True)
        self.attn_dropout = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(hidden_size)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, 4 * hidden_size), nn.GELU(),
            nn.Linear(4 * hidden_size, hidden_size), nn.Dropout(dropout),
        )

    def forward(self, x, attn_mask=None, key_padding_mask=None):
        normed = self.norm1(x)
        attn_out, _ = self.attn(
            normed, normed, normed,
            attn_mask=attn_mask,
            key_padding_mask=key_padding_mask,
            need_weights=False,
        )
        x = x + self.attn_dropout(attn_out)
        x = x + self.ffn(self.norm2(x))
        return x


block = TransformerBlock(128, 8, dropout=0.0)
block_out = block(mha_x, attn_mask=mha_causal_mask)
assert block_out.shape == mha_x.shape
print("block preserves:", block_out.shape)


### 🔥 Interview note

Most Transformer blocks preserve the main hidden-state shape: `(B,T,D) → (B,T,D)`. That invariance makes stacking easy.


## 23. Stacking Transformer Blocks

`nn.ModuleList` registers child modules so parameters appear in `model.parameters()`, move devices, save, and train correctly. A plain Python list does not register layers as expected.

### 🔥 Interview Useful


In [ ]:
layers = nn.ModuleList([TransformerBlock(128, 8, dropout=0.0) for _ in range(4)])
stack_x = mha_x
for layer in layers:
    stack_x = layer(stack_x, attn_mask=mha_causal_mask)
assert stack_x.shape == (4, 10, 128)
print("registered layer count:", len(layers))


**Debugging exercise:** `self.layers = [Block(...) for _ in range(4)]` produces outputs, but why may training/checkpointing fail? <details><summary>Answer</summary>The modules are not registered. Use `nn.ModuleList`.</details>


## 24. Output Projection and Language Model Logits

### 🔥 Interview Essential for LLMs

The LM head maps every hidden vector to one raw score per vocabulary token:

```text
hidden states (B,T,D) → Linear(D,V) → logits (B,T,V)
```

For `(8,128,50000)`, `logits[0,10]` is the vector of 50,000 vocabulary scores at position 10 of sequence 0. Keep these as raw logits for `CrossEntropyLoss`; do not apply softmax first.


In [ ]:
B, T, D, V = 2, 7, 32, 100
hidden = torch.randn(B, T, D)
lm_head = nn.Linear(D, V)
logits = lm_head(hidden)
assert logits.shape == (2, 7, 100)


## 25. Next-Token Prediction

### 🔥 Interview Essential

At every position, a causal LM predicts the following token. Synthetic shift:


In [ ]:
sequence = torch.tensor([[10, 20, 30, 40, 50]])
input_ids = sequence[:, :-1]
targets = sequence[:, 1:]
print("input: ", input_ids.tolist())
print("target:", targets.tolist())
assert input_ids.tolist() == [[10, 20, 30, 40]]
assert targets.tolist() == [[20, 30, 40, 50]]


For logits `(B,T,V)` and targets `(B,T)`, flatten to `(B×T,V)` and `(B×T)`. Each token position is one multiclass classification example.


In [ ]:
B, T, V = 3, 5, 11
toy_logits = torch.randn(B, T, V)
toy_targets = torch.randint(0, V, (B, T))
loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(toy_logits.reshape(-1, V), toy_targets.reshape(-1))
assert loss.ndim == 0
print("token-level loss:", loss.item())


### Loss-shape drills — 🟡 Medium

1. `(8,128,50000)` logits flatten to `(1024,50000)`.
2. `(8,128)` targets flatten to `(1024,)`.
3. `(4,19,128)` logits represent how many examples? **76**.
4. Should logits be probabilities? **No; CrossEntropyLoss expects raw logits.**
5. Which axes combine? **batch and sequence**.
6. What must targets contain? **integer class IDs in `[0,V)`, typically `long`.**


## 26. A Full Tiny Language Model Skeleton


In [ ]:
class TinyLanguageModel(nn.Module):
    def __init__(self, vocab_size=100, hidden_size=32, num_heads=4, num_layers=2, max_seq_len=32, dropout=0.0):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_embedding = nn.Embedding(vocab_size, hidden_size)
        self.position_embedding = nn.Embedding(max_seq_len, hidden_size)
        self.blocks = nn.ModuleList([
            TransformerBlock(hidden_size, num_heads, dropout) for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_ids):
        B, T = input_ids.shape
        if T > self.max_seq_len:
            raise ValueError(f"sequence length {T} exceeds max_seq_len {self.max_seq_len}")
        positions = torch.arange(T, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        causal_mask = torch.triu(
            torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1
        )
        for block in self.blocks:
            x = block(x, attn_mask=causal_mask)
        return self.lm_head(self.final_norm(x))


tiny_model = TinyLanguageModel()
tiny_tokens = torch.randint(0, 100, (4, 16))
tiny_logits = tiny_model(tiny_tokens)
assert tiny_logits.shape == (4, 16, 100)
print(tiny_logits.shape)


## 27. Parameter Counting

Main sources are token and position embeddings, Q/K/V and attention output projections, two FFN linear layers, LayerNorms, and the LM head.

For `Linear(in,out)`, parameters are `in×out + out` including bias. For `D=768`, each `Linear(768,768)` has `768×768+768`; combined Q/K/V has three times that, and attention output adds one more. For an FFN `768→3072→768`, count both weights and biases. FFNs often hold a large fraction of block parameters.


In [ ]:
D, FFD = 768, 3072
one_projection = D * D + D
qkv = 3 * one_projection
attention_four_projections = 4 * one_projection
ffn_parameters = D * FFD + FFD + FFD * D + D
print(f"one projection: {one_projection:,}")
print(f"QKV: {qkv:,}")
print(f"attention QKV+out: {attention_four_projections:,}")
print(f"FFN: {ffn_parameters:,}")
print(f"tiny model: {sum(p.numel() for p in tiny_model.parameters()):,}")


**Weight tying (awareness):** some LMs reuse the token embedding matrix for the output projection, reducing parameter count. It is not implemented here.


## 28. Attention Complexity and Sequence Length

### 🔥 Interview Essential for LLM roles

Each self-attention score matrix is `(T,T)`, so its storage and the sequence-length dependence of score computation are quadratic: `O(T²)`. This does **not** claim that every part of the Transformer has the same complexity.

- `T=1,000` → about 1 million pairwise positions per sequence/head.
- `T=2,000` → about 4 million.

Longer sequences therefore create larger attention matrices and can cause GPU OOM. Doubling `T` roughly quadruples attention-matrix elements.


## 29. KV Cache Concept

### 🔥 Interview Useful for LLM inference

During autoregressive generation, previously computed keys and values can be cached rather than recomputed for every new token. This speeds decode but consumes additional memory.

- **Prefill:** process all prompt tokens, creating initial K/V state.
- **Decode:** generate one token at a time and append its K/V.
- **Training:** usually predicts next tokens for many positions in parallel.
- **Generation:** autoregressively produces one new token at a time.

This is a conceptual treatment; a production KV cache is outside scope.


## 30. Inference, Greedy Selection, Sampling, and Dropout


In [ ]:
tiny_model.eval()  # disables dropout; use model.train() during training
with torch.no_grad():
    inference_logits = tiny_model(tiny_tokens)
    next_token_logits = inference_logits[:, -1, :]  # (B,V)
    greedy_token = next_token_logits.argmax(dim=-1)  # (B,)
    probs = torch.softmax(next_token_logits / 1.0, dim=-1)
    sampled_token = torch.multinomial(probs, num_samples=1)  # (B,1)
assert greedy_token.shape == (4,)
assert sampled_token.shape == (4, 1)
print(greedy_token.shape, sampled_token.shape)


Greedy decoding chooses the largest logit. Sampling draws from probabilities. Dividing logits by temperature below 1 sharpens the distribution; temperature above 1 flattens it. Training mode enables dropout; evaluation mode disables it. `torch.no_grad()` also avoids building an autograd graph for inference.


## 31. Common Mistakes

1. **Confusing `(B,T,D)` with `(T,B,D)`:** Check `batch_first`.
2. **`D=100,H=8`:** Hidden size is not divisible by heads.
3. **Scaling split-head scores by `sqrt(D)`:** Use per-head `sqrt(Dh)`.
4. **Softmax over `dim=1` for `(B,T,T)`:** Usually normalize keys with `dim=-1`.
5. **Using `Q @ K`:** Transpose K's last two axes.
6. **Masking the lower triangle:** That may hide the past; validate direction.
7. **Reversing padding-mask semantics:** In these APIs, boolean True means blocked.
8. **Dropping the residual path:** Add sublayer output back to input.
9. **`LayerNorm(T)`:** Normalize hidden size: `LayerNorm(D)`.
10. **Flattening `(B,T,D)` before blocks:** Preserve token-position structure.
11. **Softmax before CrossEntropyLoss:** Pass raw logits.
12. **Targets identical to inputs:** Shift target one position forward.
13. **Passing `(B,T,V)` directly as CE class-first input:** Flatten to `(B*T,V)`.
14. **Layers stored in a plain list:** Use `nn.ModuleList`.
15. **No causal mask in decoder self-attention:** Future information leaks.


## 32. Debugging Exercises

Diagnose each before opening the answer.

1. `scores = Q @ K` for Q/K `(4,8,16)` fails.

<details><summary>Expected diagnosis</summary>Transpose K: `Q @ K.transpose(-2,-1)`.</details>
2. `softmax(scores, dim=1)` for `(B,T,T)`.

<details><summary>Expected diagnosis</summary>Use `dim=-1` to normalize keys.</details>
3. `D=768,H=10`.

<details><summary>Expected diagnosis</summary>768 is not divisible by 10.</details>
4. CE receives logits `(8,128,50000)`, targets `(8,128)`.

<details><summary>Expected diagnosis</summary>Flatten to `(1024,50000)` and `(1024,)`.</details>
5. Embedding receives float IDs.

<details><summary>Expected diagnosis</summary>IDs must normally be `torch.long`.</details>
6. Position tensor is longer than the token sequence.

<details><summary>Expected diagnosis</summary>Create `arange(T)` from current T.</details>
7. Causal attention can see future tokens.

<details><summary>Expected diagnosis</summary>Mask upper triangle above diagonal and test outputs.</details>
8. Every attention row sums across queries, not keys.

<details><summary>Expected diagnosis</summary>Softmax axis is wrong.</details>
9. `view` fails after transpose.

<details><summary>Expected diagnosis</summary>Use `reshape` or `.contiguous().view(...)`.</details>
10. Combined heads remain `(B,H,T,Dh)`.

<details><summary>Expected diagnosis</summary>Transpose to `(B,T,H,Dh)`, then reshape.</details>
11. MHA expects `(T,B,D)` unexpectedly.

<details><summary>Expected diagnosis</summary>Set `batch_first=True` or transpose input.</details>
12. `LayerNorm(128)` receives final dimension 64.

<details><summary>Expected diagnosis</summary>normalized_shape must match the last dimension.</details>
13. Transformer parameters omit stacked blocks.

<details><summary>Expected diagnosis</summary>Use `ModuleList`, not a Python list.</details>
14. Loss targets are float.

<details><summary>Expected diagnosis</summary>Class-index targets must be long.</details>
15. Loss target IDs include V.

<details><summary>Expected diagnosis</summary>Valid target classes are `0..V-1`.</details>
16. Dropout makes causality comparison nondeterministic.

<details><summary>Expected diagnosis</summary>Call `eval()` before comparison.</details>
17. Generation tracks gradients and grows memory.

<details><summary>Expected diagnosis</summary>Use `eval()` and `torch.no_grad()`.</details>
18. Padding mask shape is `(T,T)`.

<details><summary>Expected diagnosis</summary>Per-batch key padding mask is `(B,T)`.</details>
19. Manual MHA scales by `sqrt(hidden_size)`.

<details><summary>Expected diagnosis</summary>After splitting use `sqrt(head_dim)`.</details>
20. Earlier logits change after a future-token edit.

<details><summary>Expected diagnosis</summary>Likely missing/wrong causal mask or active dropout.</details>


## 33. Attention Shape Exercises

1. Q/K `(32,12,128,64)` → scores <details><summary>Answer</summary>`(32,12,128,128)`</details>
2. weights `(8,8,100,100)`, V `(8,8,100,64)` → output <details><summary>Answer</summary>`(8,8,100,64)`</details>
3. 8 heads × 64 features → D <details><summary>Answer</summary>512</details>
4. Q `(2,4,10,16)`, K `(2,4,20,16)` → scores <details><summary>Answer</summary>`(2,4,10,20)`</details>
5. scores `(2,4,10,20)`, V `(2,4,20,16)` → output <details><summary>Answer</summary>`(2,4,10,16)`</details>
6. split `(B,T,768)` with H=12 <details><summary>Answer</summary>`(B,12,T,64)`</details>
7. combine `(3,6,7,16)` <details><summary>Answer</summary>`(3,7,96)`</details>
8. self-attention input `(4,9,32)` → scores before heads <details><summary>Answer</summary>`(4,9,9)`</details>
9. MHA output for `(4,9,32)` <details><summary>Answer</summary>`(4,9,32)`</details>
10. default averaged MHA weights, B=4,T=9 <details><summary>Answer</summary>`(4,9,9)`</details>
11. per-head MHA weights, H=4 <details><summary>Answer</summary>`(4,4,9,9)`</details>
12. tokens `(16,128)` through Embedding(V,512) <details><summary>Answer</summary>`(16,128,512)`</details>
13. positions `arange(128)` through Embedding(1024,512) <details><summary>Answer</summary>`(128,512)`</details>
14. add previous two embeddings <details><summary>Answer</summary>`(16,128,512)`</details>
15. Linear(512,2048) on `(16,128,512)` <details><summary>Answer</summary>`(16,128,2048)`</details>
16. LM head Linear(512,30000) <details><summary>Answer</summary>`(16,128,30000)`</details>
17. last-position logits <details><summary>Answer</summary>`(16,30000)`</details>
18. argmax last logits <details><summary>Answer</summary>`(16,)`</details>
19. targets for input `(16,127)` <details><summary>Answer</summary>`(16,127)`</details>
20. flatten logits `(16,127,30000)` <details><summary>Answer</summary>`(2032,30000)`</details>
21. flatten targets `(16,127)` <details><summary>Answer</summary>`(2032,)`</details>
22. causal mask at T=127 <details><summary>Answer</summary>`(127,127)`</details>
23. padding mask for tokens `(16,127)` <details><summary>Answer</summary>`(16,127)`</details>
24. cross Q length 5, K length 30, B=2,H=8 <details><summary>Answer</summary>scores `(2,8,5,30)`</details>
25. cross-attention per-head output with Dh=64 <details><summary>Answer</summary>`(2,8,5,64)`</details>


## 34. Interview Questions

Say the short answer aloud first, then open the detail.

### 1. What is the standard batch-first hidden-state shape?

**Short Interview Answer:** `(B,T,D)`.

<details><summary>Detailed Explanation</summary>Batch, token position, then hidden features.</details>

### 2. What does `nn.Embedding` do?

**Short Interview Answer:** Maps integer IDs to learned vectors.

<details><summary>Detailed Explanation</summary>It indexes a learned `(V,D)` table and maps `(B,T)` to `(B,T,D)`.</details>

### 3. Why positional information?

**Short Interview Answer:** Attention alone does not encode order.

<details><summary>Detailed Explanation</summary>Position representations distinguish permutations of the same tokens.</details>

### 4. What are Q, K, V?

**Short Interview Answer:** Learned projections used to score and mix tokens.

<details><summary>Detailed Explanation</summary>Queries compare with keys; weights then combine values. The intuition is not a literal semantic guarantee.</details>

### 5. How are attention scores computed?

**Short Interview Answer:** `Q @ K.transpose(-2,-1)`.

<details><summary>Detailed Explanation</summary>The inner feature dimension contracts, producing query-by-key scores.</details>

### 6. What shape is QKᵀ in single-head self-attention?

**Short Interview Answer:** `(B,T,T)`.

<details><summary>Detailed Explanation</summary>Each query position scores all T key positions.</details>

### 7. Why divide by sqrt(d_k)?

**Short Interview Answer:** To control score magnitudes before softmax.

<details><summary>Detailed Explanation</summary>Dot-product magnitude tends to grow with dimension, which can make softmax overly saturated.</details>

### 8. Why softmax over keys?

**Short Interview Answer:** Each query needs a distribution over key positions.

<details><summary>Detailed Explanation</summary>That is the final dimension in the shown score layouts.</details>

### 9. What does the attention matrix represent?

**Short Interview Answer:** Per-query weights over keys.

<details><summary>Detailed Explanation</summary>Each row controls how values are combined for one query.</details>

### 10. What is self-attention?

**Short Interview Answer:** Q, K, and V come from the same sequence.

<details><summary>Detailed Explanation</summary>It lets positions exchange information within one representation.</details>

### 11. What is cross-attention?

**Short Interview Answer:** Q comes from one sequence; K/V from another.

<details><summary>Detailed Explanation</summary>Its score shape is `(B,H,T_query,T_context)`.</details>

### 12. What is causal attention?

**Short Interview Answer:** Attention that blocks future positions.

<details><summary>Detailed Explanation</summary>It preserves autoregressive next-token prediction.</details>

### 13. Why do decoder LMs need a causal mask?

**Short Interview Answer:** To prevent future-token leakage.

<details><summary>Detailed Explanation</summary>Position t must predict using only positions at or before t.</details>

### 14. What is a padding mask?

**Short Interview Answer:** A per-example mask for padded keys.

<details><summary>Detailed Explanation</summary>It prevents meaningless padding representations from contributing.</details>

### 15. Causal vs padding mask?

**Short Interview Answer:** Future structure vs padded data.

<details><summary>Detailed Explanation</summary>They address different constraints and can be combined.</details>

### 16. What is multi-head attention?

**Short Interview Answer:** Parallel attention over split feature subspaces.

<details><summary>Detailed Explanation</summary>D is split into H heads of dimension Dh, then outputs are recombined.</details>

### 17. Why multiple heads?

**Short Interview Answer:** They provide multiple learned attention subspaces.

<details><summary>Detailed Explanation</summary>Different projections can learn different interaction patterns in parallel.</details>

### 18. What is head dimension?

**Short Interview Answer:** `Dh=D/H`.

<details><summary>Detailed Explanation</summary>It is the key/query/value feature width inside one head.</details>

### 19. What divisibility rule applies?

**Short Interview Answer:** `D % H == 0`.

<details><summary>Detailed Explanation</summary>Equal-sized heads require an integral Dh.</details>

### 20. Q shape after splitting?

**Short Interview Answer:** `(B,H,T,Dh)`.

<details><summary>Detailed Explanation</summary>Reshape to expose H and Dh, then transpose heads before sequence.</details>

### 21. How are heads combined?

**Short Interview Answer:** Transpose to `(B,T,H,Dh)`, reshape to `(B,T,D)`.

<details><summary>Detailed Explanation</summary>An output projection usually follows.</details>

### 22. What does `nn.MultiheadAttention` do?

**Short Interview Answer:** Implements projected multi-head attention.

<details><summary>Detailed Explanation</summary>It accepts query/key/value plus optional structural and padding masks.</details>

### 23. Why `batch_first=True`?

**Short Interview Answer:** It makes inputs `(B,T,D)`.

<details><summary>Detailed Explanation</summary>That convention matches most data pipelines in this course.</details>

### 24. What is a residual connection?

**Short Interview Answer:** Input plus sublayer output.

<details><summary>Detailed Explanation</summary>It preserves a direct information and gradient path.</details>

### 25. Why are residuals important?

**Short Interview Answer:** They help optimization and preserve information.

<details><summary>Detailed Explanation</summary>Deep stacks become easier to train.</details>

### 26. Why LayerNorm?

**Short Interview Answer:** It stabilizes token hidden features.

<details><summary>Detailed Explanation</summary>It normalizes the hidden dimension independently per token.</details>

### 27. BatchNorm vs LayerNorm?

**Short Interview Answer:** LayerNorm does not aggregate batch statistics.

<details><summary>Detailed Explanation</summary>It fits variable sequences and behaves consistently for small/inference batches.</details>

### 28. What is the Transformer FFN?

**Short Interview Answer:** A per-token nonlinear projection, often D→4D→D.

<details><summary>Detailed Explanation</summary>Linear acts on the final axis while leaving B and T unchanged.</details>

### 29. Why does the FFN preserve sequence length?

**Short Interview Answer:** It operates independently on each token's features.

<details><summary>Detailed Explanation</summary>No operation mixes or removes token positions.</details>

### 30. What shape does a block preserve?

**Short Interview Answer:** Usually `(B,T,D)`.

<details><summary>Detailed Explanation</summary>Residual additions require matching input/output shapes.</details>

### 31. What does a language model output?

**Short Interview Answer:** Raw vocabulary logits at every position.

<details><summary>Detailed Explanation</summary>Each position receives V scores.</details>

### 32. LM logit shape?

**Short Interview Answer:** `(B,T,V)`.

<details><summary>Detailed Explanation</summary>Vocabulary is the final class dimension.</details>

### 33. Why is vocabulary last?

**Short Interview Answer:** Linear maps the final D axis to V.

<details><summary>Detailed Explanation</summary>It also makes each `[b,t]` slice a class-score vector.</details>

### 34. How is next-token prediction trained?

**Short Interview Answer:** Predict shifted targets at all positions.

<details><summary>Detailed Explanation</summary>Teacher-forced training parallelizes positions behind a causal mask.</details>

### 35. Why shift inputs and targets?

**Short Interview Answer:** Each input prefix predicts its following token.

<details><summary>Detailed Explanation</summary>Without shifting, the labels describe the current rather than next token.</details>

### 36. How compute CE for `(B,T,V)`?

**Short Interview Answer:** Reshape to `(B*T,V)` and targets to `(B*T,)`.

<details><summary>Detailed Explanation</summary>Each position becomes one classification example.</details>

### 37. Why quadratic attention scaling?

**Short Interview Answer:** Scores contain T×T query-key pairs.

<details><summary>Detailed Explanation</summary>Doubling sequence length quadruples score-matrix elements.</details>

### 38. What is a KV cache?

**Short Interview Answer:** Stored past keys and values for decoding.

<details><summary>Detailed Explanation</summary>New tokens reuse earlier K/V instead of recomputing them.</details>

### 39. Why does a KV cache speed generation?

**Short Interview Answer:** It avoids repeated past K/V computation.

<details><summary>Detailed Explanation</summary>Decode processes the new query against cached context.</details>

### 40. KV-cache trade-off?

**Short Interview Answer:** More speed for more memory.

<details><summary>Detailed Explanation</summary>Cache size grows with layers, context length, heads, and head dimension.</details>

### 41. Prefill vs decode?

**Short Interview Answer:** Prompt processing vs token-by-token generation.

<details><summary>Detailed Explanation</summary>Prefill creates cache state; decode extends it.</details>

### 42. What is greedy decoding?

**Short Interview Answer:** Choose argmax at the last position.

<details><summary>Detailed Explanation</summary>It is deterministic but may be less diverse than sampling.</details>

### 43. What does temperature do?

**Short Interview Answer:** Rescales logits before softmax.

<details><summary>Detailed Explanation</summary>Lower is sharper; higher is flatter.</details>


## 35. Knowledge Check Quiz

Choose before checking the separate key.

**1. For `(16,128,768)` and H=12, Dh?**

A. 12
B. 64
C. 128
D. 768

**2. Q/K `(B,H,T,Dh)` gives score shape?**

A. (B,H,Dh,Dh)
B. (B,T,T)
C. (B,H,T,T)
D. (B,H,T,Dh)

**3. Why causal mask?**

A. Hide padding
B. Block future tokens
C. Normalize
D. Reduce V

**4. Token IDs dtype?**

A. float32
B. bool
C. long
D. complex

**5. Embedding maps `(B,T)` to?**

A. (B,D)
B. (T,D)
C. (B,T,D)
D. (B,T,V)

**6. Softmax attention axis?**

A. batch
B. heads
C. queries
D. keys / last

**7. Single-head score shape?**

A. (B,T,T)
B. (B,T,D)
C. (T,D)
D. (B,D,D)

**8. Correct scale after splitting?**

A. sqrt(D)
B. sqrt(H)
C. sqrt(Dh)
D. T

**9. Causal mask construction?**

A. triu diagonal=1
B. tril diagonal=-1
C. eye only
D. zeros

**10. Padding mask shape?**

A. (T,T)
B. (B,T)
C. (B,D)
D. (H,T,T)

**11. Self-attention call?**

A. attn(x,y,y)
B. attn(x,x,x)
C. attn(q,q,v)
D. attn(x)

**12. Cross-attention K/V source?**

A. query sequence
B. context sequence
C. vocabulary
D. positions

**13. Can D=100,H=8?**

A. yes
B. no
C. only eval
D. only CPU

**14. Split shape?**

A. (B,T,H,Dh)
B. (B,H,T,Dh)
C. (H,B,Dh,T)
D. (B,D,T)

**15. Default averaged MHA weights?**

A. (B,H,T,T)
B. (B,T,T)
C. (T,T)
D. (B,T,D)

**16. LayerNorm argument?**

A. B
B. T
C. D
D. V

**17. FFN changes which axis?**

A. batch
B. sequence
C. feature
D. head count only

**18. Block usual output shape?**

A. (B,T,D)
B. (B,T,4D)
C. (B,D)
D. (T,B,D)

**19. Registered stack container?**

A. tuple
B. dict
C. list
D. ModuleList

**20. LM logits shape?**

A. (B,V)
B. (B,T,D)
C. (B,T,V)
D. (T,V)

**21. CE input after flatten?**

A. (B*T,V)
B. (B,V*T)
C. (B*T,D)
D. (T,V)

**22. CE targets dtype?**

A. bool
B. long class IDs
C. float probabilities required
D. complex

**23. Apply softmax before CE?**

A. always
B. never for standard raw logits
C. only CPU
D. only training

**24. Attention matrix sequence dependence?**

A. O(1)
B. O(T)
C. O(T²)
D. O(log T)

**25. KV cache primarily helps?**

A. training backprop
B. autoregressive decode
C. tokenization
D. LayerNorm

**26. KV-cache cost?**

A. accuracy
B. memory
C. vocabulary
D. labels

**27. Greedy token shape from `(B,V)`?**

A. (B,1,V)
B. (B,)
C. (V,)
D. scalar

**28. Lower temperature?**

A. flatter
B. sharper
C. no effect
D. more heads

**29. Eval mode affects?**

A. dropout
B. parameter count
C. tensor dtype
D. vocabulary

**30. Future-edit causality test needs?**

A. train mode
B. eval mode
C. no mask
D. BatchNorm

### Answer key

1-B, 2-C, 3-B, 4-C, 5-C, 6-D, 7-A, 8-C, 9-A, 10-B, 11-B, 12-B, 13-B, 14-B, 15-B, 16-C, 17-C, 18-A, 19-D, 20-C, 21-A, 22-B, 23-B, 24-C, 25-B, 26-B, 27-B, 28-B, 29-A, 30-B


## 36. Write From Memory

Write each without looking back; validate shapes.

1. Create `Embedding(30000,512)`.
2. Generate positions for current T on input device.
3. Create Q/K/V `Linear(D,D)` projections.
4. Compute QKᵀ.
5. Scale scores by sqrt(d_k).
6. Softmax scores over keys.
7. Multiply weights by V.
8. Create an upper-triangular causal mask.
9. Fill future scores with `-inf`.
10. Split `(B,T,D)` to `(B,H,T,Dh)`.
11. Combine `(B,H,T,Dh)` to `(B,T,D)`.
12. Create batch-first MHA.
13. Create LayerNorm for D=768.
14. Create FFN 768→3072→768.
15. Create an LM head D→V.
16. Shift tokens into input and target.
17. Reshape LM logits and targets for CE.
18. Build a dynamic `(T,T)` causal mask.
19. Take last-position logits and greedy argmax.
20. Count all trainable model parameters.


In [ ]:
# Compact executable validation target for the memory prompts
B, T, D, H, V = 2, 6, 32, 4, 100
ids = torch.randint(0, V, (B, T))
emb = nn.Embedding(V, D)
pos_emb = nn.Embedding(32, D)
x = emb(ids) + pos_emb(torch.arange(T, device=ids.device))
q, k, v = nn.Linear(D, D)(x), nn.Linear(D, D)(x), nn.Linear(D, D)(x)
Dh = D // H
q = q.reshape(B, T, H, Dh).transpose(1, 2)
k = k.reshape(B, T, H, Dh).transpose(1, 2)
v = v.reshape(B, T, H, Dh).transpose(1, 2)
s = (q @ k.transpose(-2, -1)) / math.sqrt(Dh)
m = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
w = torch.softmax(s.masked_fill(m, float("-inf")), dim=-1)
o = (w @ v).transpose(1, 2).reshape(B, T, D)
assert o.shape == (B, T, D)


## 37. Transformer Syntax Flashcards

1. **Token IDs → embeddings?** <details><summary>Recall</summary>`nn.Embedding(V,D)(ids)`</details>
2. **Position IDs?** <details><summary>Recall</summary>`torch.arange(T, device=ids.device)`</details>
3. **Q projection?** <details><summary>Recall</summary>`Q = q_proj(x)`</details>
4. **Attention scores?** <details><summary>Recall</summary>`Q @ K.transpose(-2,-1)`</details>
5. **Scale?** <details><summary>Recall</summary>`/ math.sqrt(head_dim)`</details>
6. **Attention probabilities?** <details><summary>Recall</summary>`torch.softmax(scores, dim=-1)`</details>
7. **Block future positions?** <details><summary>Recall</summary>`scores.masked_fill(mask, float('-inf'))`</details>
8. **Causal mask?** <details><summary>Recall</summary>`torch.triu(torch.ones(T,T,dtype=torch.bool), diagonal=1)`</details>
9. **Padding mask from ID 0?** <details><summary>Recall</summary>`ids == 0`</details>
10. **Attention output?** <details><summary>Recall</summary>`weights @ V`</details>
11. **Split heads?** <details><summary>Recall</summary>`x.reshape(B,T,H,Dh).transpose(1,2)`</details>
12. **Combine heads?** <details><summary>Recall</summary>`x.transpose(1,2).reshape(B,T,D)`</details>
13. **Batch-first MHA?** <details><summary>Recall</summary>`nn.MultiheadAttention(D,H,batch_first=True)`</details>
14. **Self-attention call?** <details><summary>Recall</summary>`attn(x,x,x)`</details>
15. **Residual?** <details><summary>Recall</summary>`x = x + sublayer_output`</details>
16. **LayerNorm?** <details><summary>Recall</summary>`nn.LayerNorm(D)`</details>
17. **FFN expansion?** <details><summary>Recall</summary>`nn.Linear(D,4*D)`</details>
18. **LM logits?** <details><summary>Recall</summary>`nn.Linear(D,V)(x)`</details>
19. **Shift targets?** <details><summary>Recall</summary>`inputs=tokens[:,:-1]; targets=tokens[:,1:]`</details>
20. **Token CE?** <details><summary>Recall</summary>`loss_fn(logits.reshape(-1,V), targets.reshape(-1))`</details>


## 38. Final Mini Transformer Challenge — 🔴 Hard / Optional

Build and run a decoder-style LM with `vocab_size=128`, `max_seq_len=32`, `hidden_size=64`, `num_heads=4`, `num_layers=2`. It needs token and position embeddings, two pre-norm causal blocks, final LayerNorm, vocabulary projection, shifted inputs/targets, CrossEntropyLoss, AdamW, a few training steps, a causality test, and greedy generation.


In [ ]:
torch.manual_seed(42)
vocab_size = 128
challenge_model = TinyLanguageModel(
    vocab_size=vocab_size, max_seq_len=32, hidden_size=64,
    num_heads=4, num_layers=2, dropout=0.0,
)
tokens = torch.randint(1, vocab_size, (256, 20))
input_ids, targets = tokens[:, :-1], tokens[:, 1:]
assert input_ids.shape == targets.shape == (256, 19)

optimizer = torch.optim.AdamW(challenge_model.parameters(), lr=1e-3)
challenge_model.train()
for step in range(3):
    optimizer.zero_grad()
    # A small slice keeps the demonstration quick on CPU.
    batch_ids, batch_targets = input_ids[:16], targets[:16]
    challenge_logits = challenge_model(batch_ids)
    challenge_loss = F.cross_entropy(
        challenge_logits.reshape(-1, vocab_size), batch_targets.reshape(-1)
    )
    challenge_loss.backward()
    optimizer.step()
    print(f"step {step + 1}: loss={challenge_loss.item():.4f}")
assert challenge_logits.shape == (16, 19, vocab_size)


### Verify causality — 🔴 Hard / Optional

Change a future token while keeping earlier tokens identical. In eval mode with dropout disabled, logits at earlier positions must stay unchanged within tolerance.


In [ ]:
challenge_model.eval()
a = torch.tensor([[1, 2, 3, 4, 5, 6]])
b = a.clone()
b[0, 4] = 99  # change position 4; positions 0..3 must be unaffected
with torch.no_grad():
    logits_a = challenge_model(a)
    logits_b = challenge_model(b)
max_early_difference = (logits_a[:, :4] - logits_b[:, :4]).abs().max().item()
print("max difference before edited future position:", max_early_difference)
assert torch.allclose(logits_a[:, :4], logits_b[:, :4], atol=1e-6)


The model trains on random IDs, so generated IDs are not expected to be meaningful. The loop demonstrates autoregressive tensor flow only.


In [ ]:
context = torch.tensor([[1, 2, 3]])
challenge_model.eval()
with torch.no_grad():
    for _ in range(5):
        generation_logits = challenge_model(context)
        next_id = generation_logits[:, -1, :].argmax(dim=-1, keepdim=True)
        context = torch.cat([context, next_id], dim=1)
print("generated IDs:", context.tolist())
assert context.shape == (1, 8)


## 39. Cheat Sheet

```python
# Embeddings and positions
token_embedding = nn.Embedding(vocab_size, hidden_size)
x = token_embedding(input_ids)
positions = torch.arange(input_ids.size(1), device=input_ids.device)
x = x + position_embedding(positions)

# Q, K, V and attention
Q, K, V = q_proj(x), k_proj(x), v_proj(x)
scores = (Q @ K.transpose(-2, -1)) / math.sqrt(head_dim)
weights = torch.softmax(scores, dim=-1)
output = weights @ V

# Causal mask
mask = torch.triu(
    torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1
)
scores = scores.masked_fill(mask, float("-inf"))

# High-level attention
attention = nn.MultiheadAttention(hidden_size, num_heads, batch_first=True)
output, weights = attention(x, x, x)

# FFN and LM head
ffn = nn.Sequential(
    nn.Linear(hidden_size, 4 * hidden_size), nn.GELU(),
    nn.Linear(4 * hidden_size, hidden_size),
)
logits = nn.Linear(hidden_size, vocab_size)(x)
loss = loss_fn(logits.reshape(-1, vocab_size), targets.reshape(-1))
```

### 🔥 Memorize These Shapes

```text
Token IDs       (B,T)
   ↓ Embedding
Hidden states   (B,T,D)
   ↓ Q/K/V
Q/K/V           (B,T,D)
   ↓ split
Heads           (B,H,T,Dh)
   ↓ QKᵀ
Scores          (B,H,T,T)
   ↓ softmax
Weights         (B,H,T,T)
   ↓ × V
Per-head output (B,H,T,Dh)
   ↓ combine
Hidden output   (B,T,D)
   ↓ residual + LayerNorm + FFN
Block output    (B,T,D)
   ↓ LM head
Logits          (B,T,V)
```

### Interview quick reference

Embedding → IDs to vectors · Q/K/V → projected attention representations · QKᵀ → pairwise scores · sqrt(d_k) → controlled score scale · softmax → weights · causal mask → hide future · multi-head → parallel subspaces · residual → input + sublayer · LayerNorm → normalize token features · FFN → per-token nonlinear transformation · LM head → vocabulary logits · KV cache → reuse past keys/values during decode.


## 40. Before Moving to `11_pytorch_interview_challenge.ipynb`

- [ ] I understand `(batch, sequence, hidden_size)`.
- [ ] I can use `nn.Embedding`.
- [ ] I can add positional embeddings.
- [ ] I understand Q, K, and V.
- [ ] I can compute QKᵀ.
- [ ] I understand attention score shapes.
- [ ] I know why attention is scaled by `sqrt(d_k)`.
- [ ] I know which dimension softmax uses.
- [ ] I can create a causal mask.
- [ ] I understand padding masks.
- [ ] I can explain self-attention.
- [ ] I can explain cross-attention.
- [ ] I understand multi-head attention.
- [ ] I can calculate head dimension.
- [ ] I can split and combine attention heads.
- [ ] I can use `nn.MultiheadAttention`.
- [ ] I understand residual connections.
- [ ] I understand LayerNorm in Transformers.
- [ ] I can build a Transformer FFN.
- [ ] I can build a simple Transformer block.
- [ ] I understand language-model logits `(B,T,V)`.
- [ ] I can shift LM inputs and targets.
- [ ] I can compute token-level CrossEntropyLoss.
- [ ] I understand quadratic attention scaling with sequence length.
- [ ] I understand KV cache conceptually.
- [ ] I can build a tiny autoregressive Transformer in PyTorch.
